# Weather Detection (Weather Type Classifier)
Predicts the weather type (**sun, rain, drizzle, fog, snow**) from precipitation, temperature, wind and month.

Dataset: Seattle weather, 2012 to 2015 (1,461 days). Run the cells from top to bottom (Runtime → Run all) and upload `weather.csv` when asked.

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, f1_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

## 2. Upload dataset
Select `weather.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 3. Load and explore the data

In [ ]:
df = pd.read_csv("weather.csv").dropna().drop_duplicates()
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

print("Dataset shape:", df.shape)
print(df["weather"].value_counts())
df.head()

In [ ]:
# Class distribution and average values for each weather type
fig, ax = plt.subplots(figsize=(6, 3.5))
df["weather"].value_counts().plot(kind="bar", color="steelblue", ax=ax)
ax.set_title("Number of days per weather type")
ax.set_ylabel("Days")
plt.xticks(rotation=0)
plt.show()

df.groupby("weather")[["precipitation", "temp_max", "temp_min", "wind"]].mean().round(2)

## 4. Feature engineering
Three extra features are created from the existing columns.

In [ ]:
df["temp_range"] = df["temp_max"] - df["temp_min"]
df["temp_avg"] = (df["temp_max"] + df["temp_min"]) / 2
df["wet"] = (df["precipitation"] > 0).astype(int)

FEATURES = ["precipitation", "temp_max", "temp_min", "wind", "month",
            "temp_range", "temp_avg", "wet"]
X, y = df[FEATURES], df["weather"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Compare models (5-fold cross-validation on training data)
The classes are imbalanced, so **macro F1** is used to pick the best model.

In [ ]:
models = {
    "Logistic Regression": Pipeline([("sc", StandardScaler()),
                                     ("m", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "KNN": Pipeline([("sc", StandardScaler()), ("m", KNeighborsClassifier(n_neighbors=7))]),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                            class_weight="balanced_subsample", random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, m in models.items():
    acc = cross_val_score(m, X_train, y_train, cv=cv, scoring="accuracy").mean()
    f1 = cross_val_score(m, X_train, y_train, cv=cv, scoring="f1_macro").mean()
    rows.append((name, acc, f1))

results = pd.DataFrame(rows, columns=["Model", "CV Accuracy", "CV Macro F1"])
results = results.sort_values("CV Macro F1", ascending=False).round(4)
results

## 6. Train the best model and test it

In [ ]:
best_name = results.iloc[0]["Model"]
best = models[best_name].fit(X_train, y_train)
pred = best.predict(X_test)

print("Best model:", best_name)
print(f"Test accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"Test macro F1: {f1_score(y_test, pred, average='macro'):.4f}\n")
print(classification_report(y_test, pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues", xticks_rotation=45)
plt.title(f"Confusion Matrix ({best_name})")
plt.show()

## 7. Which features matter most?

In [ ]:
if hasattr(best, "feature_importances_"):
    imp = pd.Series(best.feature_importances_, index=FEATURES).sort_values()
    imp.plot(kind="barh", color="steelblue", figsize=(6, 4))
    plt.title("Feature importance")
    plt.show()

## 8. Predict the weather

In [ ]:
def predict_weather(precipitation, temp_max, temp_min, wind, month):
    row = pd.DataFrame([[precipitation, temp_max, temp_min, wind, month]],
                       columns=["precipitation", "temp_max", "temp_min", "wind", "month"])
    row["temp_range"] = row["temp_max"] - row["temp_min"]
    row["temp_avg"] = (row["temp_max"] + row["temp_min"]) / 2
    row["wet"] = (row["precipitation"] > 0).astype(int)
    return best.predict(row[FEATURES])[0]

samples = [
    ("Rain 10.9 mm, 10.6/2.8 C, wind 4.5, January", (10.9, 10.6, 2.8, 4.5, 1)),
    ("Dry, 28/14 C, wind 2.5, July",                 (0.0, 28.0, 14.0, 2.5, 7)),
    ("Dry, 7.2/1.1 C, wind 1.8, December",           (0.0, 7.2, 1.1, 1.8, 12)),
    ("Light precipitation, 0.5/-3 C, January",       (5.0, 0.5, -3.0, 3.0, 1)),
]
for text, vals in samples:
    print(f"{text:<48} -> {predict_weather(*vals)}")

## 9. Try your own values
Order: precipitation (mm), max temp (C), min temp (C), wind speed, month (1-12).

In [ ]:
print(predict_weather(2.0, 12.0, 6.0, 3.5, 11))

## 10. Download the trained model (optional)

In [ ]:
import joblib
joblib.dump(best, "weather_model.joblib")
files.download("weather_model.joblib")